# Search Engine TREC 2020 Deep Learning Track

Course: Multimedia Information Retrieval and Computer Vision, University of Pisa
Academic Year: 2026/2027

## 1. Set-up

### 1.1 package installation with pinned versions

In [1]:
import os
import sys

from pathlib import Path

In [2]:
!{sys.executable} -m pip install nltk==3.10.4

In [3]:
!{sys.executable} -m pip install numpy==2.5.4

In [4]:
!{sys.executable} -m pip install matplotlib==3.11.2

In [5]:
!{sys.executable} -m pip install ir-datasets==0.6.3

In [6]:
!{sys.executable} -m pip install psutil==7.2.2

In [7]:
!{sys.executable} -m pip install py-cpuinfo==9.0.0

### 1.2 Google Drive mounting

In [8]:
# IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [9]:
if not IN_COLAB:
    !{sys.executable} -m pip install PyDrive2==1.21.3

In [10]:
if not IN_COLAB:
    !{sys.executable} -m pip install google-auth==2.62.0

In [11]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

    # Colab uses 'MyDrive' in recent versions, but older notebooks use 'My Drive'
    DRIVE_ROOT = '/content/drive/MyDrive'
    if not os.path.isdir(DRIVE_ROOT):
        DRIVE_ROOT = '/content/drive/My Drive'

    FOLDER = 'AIDE/MIR/Search Engine TREC 2020 Deep Learning Track'
    folder_path = os.path.join(DRIVE_ROOT, FOLDER)

    print(f"Exists: {os.path.exists(folder_path)}")
    if os.path.exists(folder_path):
        print(os.listdir(folder_path)[:10])
else:
    from pydrive2.auth import GoogleAuth
    from pydrive2.drive import GoogleDrive

    # https://drive.google.com/drive/folders/1Gv83xh2BntmhyqXRdWYVXw3RLWUobBLS

    # Authenticate and create the PyDrive client
    gauth = GoogleAuth()
    # This will open a browser for first-time authentication
    gauth.LocalWebserverAuth() 

    drive = GoogleDrive(gauth)

    FOLDER_PATH = ['AIDE', 'MIR', 'Search Engine TREC 2020 Deep Learning Track']

    def resolve_folder_id(drive, parts, parent='root'):
        """Walk the path one folder at a time, returning the final folder's ID."""
        for part in parts:
            query = (
                f"'{parent}' in parents and title = '{part}' "
                f"and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
            )
            matches = drive.ListFile({'q': query}).GetList()
            if not matches:
                raise FileNotFoundError(f"'{part}' not found under parent {parent}")
            parent = matches[0]['id']
        return parent

    folder_id = resolve_folder_id(drive, FOLDER_PATH)
    print(f"Folder ID: {folder_id}")


    files = drive.ListFile({'q': f"'{folder_id}' in parents and trashed = false"}).GetList()
    for f in files:
        print(f['title'], f['id'])

    # Download one file
    target = next(f for f in files if f['title'].endswith('.txt'))
    target.GetContentFile(target['title'])

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=328948873835-vsili92ed42o61333v7do09h411mdvpp.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=online&response_type=code

Authentication successful.
Folder ID: 1Gv83xh2BntmhyqXRdWYVXw3RLWUobBLS
49960.txt 1CLTZqvi8ZpL7TgypFDdYo1ACW2kZDNrA


### 1.3 Runtime characteristics

In [12]:
import platform
import sys
import os
import psutil

def print_runtime_characteristics():
    """Prints a summary of the current Python and hardware environment."""
    print("=" * 50)
    print("       RUNTIME ENVIRONMENT REPORT")
    print("=" * 50)

    # --- Python Information ---
    print("\n--- Python ---")
    print(f"Version:      {sys.version.split()[0]}")  # e.g., 3.11.5
    print(f"Implementation: {platform.python_implementation()}")  # e.g., CPython
    print(f"Executable:   {sys.executable}")

    # --- Operating System ---
    print("\n--- Operating System ---")
    print(f"System:       {platform.system()} {platform.release()}")  # e.g., Linux 5.15
    print(f"Machine:      {platform.machine()}")  # e.g., x86_64
    print(f"Platform:     {platform.platform()}")  # Full platform string

    # --- CPU Information ---
    print("\n--- CPU ---")
    print(f"Physical Cores: {psutil.cpu_count(logical=False)}")
    print(f"Logical Cores:  {psutil.cpu_count(logical=True)}")
    # The line below attempts to get the CPU brand string, but may fail on some systems
    try:
        import cpuinfo
        print(f"Processor:      {cpuinfo.get_cpu_info()['brand_raw']}")
    except ImportError:
        pass  # cpuinfo is optional

    # --- Memory (RAM) ---
    print("\n--- Memory ---")
    svmem = psutil.virtual_memory()
    # Helper to format bytes into readable units (GB)
    def bytes_to_gb(bytes_val):
        return f"{bytes_val / (1024**3):.2f} GB"

    print(f"Total RAM:     {bytes_to_gb(svmem.total)}")
    print(f"Available:     {bytes_to_gb(svmem.available)}")
    print(f"Used:          {bytes_to_gb(svmem.used)} ({svmem.percent}%)")

    print("\n" + "=" * 50)

# Call the function to print the report
print_runtime_characteristics()

       RUNTIME ENVIRONMENT REPORT

--- Python ---
Version:      3.15.0
Implementation: CPython
Executable:   /Users/adrienkt/DataspellProjects/search-engine-trec-2020-deep-learning-track/.venv/bin/python

--- Operating System ---
System:       Darwin 27.0.0
Machine:      arm64
Platform:     macOS-27.0.1-arm64-arm-64bit-Mach-O

--- CPU ---
Physical Cores: 8
Logical Cores:  8
Processor:      Apple M2

--- Memory ---
Total RAM:     8.00 GB
Available:     1.07 GB
Used:          3.61 GB (86.6%)



## 2. Data

### 2.0 Configuration

In [13]:
# Dossier Drive contenant vos données (ID résolu précédemment)
DRIVE_FOLDER_ID = '1Gv83xh2BntmhyqXRdWYVXw3RLWUobBLS'

# Répertoires de travail
if IN_COLAB:
    WORK = Path('/content')
    DRIVE_MOUNT = Path('/content/drive/MyDrive')
else:
    WORK = Path.cwd() / 'work'
    WORK.mkdir(exist_ok=True)
    DRIVE_MOUNT = None  # accès via PyDrive2, pas de mount local

DATA_DIR = WORK / 'data'
DATA_DIR.mkdir(exist_ok=True)

COLLECTION_TGZ = DATA_DIR / 'collection.tar.gz'
COLLECTION_TSV = DATA_DIR / 'collection.tsv'
PID_MAP_FILE   = DATA_DIR / 'pid_to_docid.npy'

print(f"Environment : {'Colab' if IN_COLAB else 'Local'}")
print(f"Work dir    : {WORK}")
print(f"Data dir    : {DATA_DIR}")

Environment : Local
Work dir    : /Users/adrienkt/DataspellProjects/search-engine-trec-2020-deep-learning-track/work
Data dir    : /Users/adrienkt/DataspellProjects/search-engine-trec-2020-deep-learning-track/work/data


### 2.1 Get Collection

In [14]:
COLLECTION_URL = 'https://msmarco.z22.web.core.windows.net/msmarcoranking/collection.tar.gz'

if not COLLECTION_TGZ.exists():
    print("Téléchargement depuis msmarco.z22.web.core.windows.net…")
    
    #if IN_COLAB:
    #    !wget -q --show-progress -O "{COLLECTION_TGZ}" "{COLLECTION_URL}"
    #else:
        # Local : curl ou urllib
    #    import urllib.request
    #    urllib.request.urlretrieve(COLLECTION_URL, COLLECTION_TGZ)
    
    !wget -q --show-progress -O "{COLLECTION_TGZ}" "{COLLECTION_URL}"
    print(f"{COLLECTION_TGZ.stat().st_size / 1e9:.2f} GB")
else:
    print(f"Collection already present : {COLLECTION_TGZ}")

Collection already present : /Users/adrienkt/DataspellProjects/search-engine-trec-2020-deep-learning-track/work/data/collection.tar.gz


Extract the file from the archive

In [15]:
import tarfile

if not COLLECTION_TSV.exists():
    print(f"Extraction de {COLLECTION_TGZ.name}…")
    with tarfile.open(COLLECTION_TGZ, 'r:gz') as tf:
        # L'archive containt one file 'collection.tsv'
        member = tf.getmember('collection.tsv')
        tf.extract(member, path=DATA_DIR)
    print(f"{COLLECTION_TSV} ({COLLECTION_TSV.stat().st_size / 1e9:.2f} GB)")
else:
    print(f"Already extracted : {COLLECTION_TSV}")

Already extracted : /Users/adrienkt/DataspellProjects/search-engine-trec-2020-deep-learning-track/work/data/collection.tsv


### 2.2 Robust parser with handling for "messy" inputs

The parser:

- opens using UTF-8 with `errors='replace'`;
- skips lines that lack a tab character or contain a non-integer PID, and keeps a count of them;
- retains empty segments after decoding (`text == ''`), assigns each a sequential `docid`, and keeps a count of them;
- explicitly stores the mapping from PID to `docid`.

In [16]:
import numpy as np
from tqdm.auto import tqdm

def parse_collection(path, max_lines=None, verbose=True):
    """
    Parse collection.tsv handling the malformed lines.

    Returns
    -------
    pid_to_docid : dict[int, int]
    docid_to_pid : np.ndarray (int64), taille = number of passages maintained
    stats        : dict avec 'kept', 'skipped', 'empty'
    """
    pid_to_docid = {}
    docid_to_pid = []
    skipped = 0
    empty = 0

    # mode texte : universal newlines + replacement decoder
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        it = enumerate(f)
        if verbose:
            it = tqdm(it, total=max_lines or None, desc='Parsing', unit='line')

        for line_no, line in it:
            if max_lines is not None and line_no >= max_lines:
                break

            # remove only the line terminator(s)
            text_line = line.rstrip('\r\n')

            # split on the FIRST tab only : preserve the tabs in the text
            parts = text_line.split('\t', 1)
            if len(parts) != 2:
                skipped += 1
                continue

            pid_str, text = parts
            try:
                pid = int(pid_str)
            except ValueError:
                skipped += 1
                continue

            docid = len(docid_to_pid)          # consecutive from 0
            pid_to_docid[pid] = docid
            docid_to_pid.append(pid)

            if text == '':                     # passage empty "after processing"
                empty += 1

    docid_to_pid = np.asarray(docid_to_pid, dtype=np.int64)
    stats = {
        'kept':    len(docid_to_pid),
        'skipped': skipped,
        'empty':   empty,
    }
    return pid_to_docid, docid_to_pid, stats

/Users/adrienkt/DataspellProjects/search-engine-trec-2020-deep-learning-track/.venv/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Parsing the development subset (100,000 lines)

In [17]:
pid_to_docid_dev, docid_to_pid_dev, stats_dev = parse_collection(
    COLLECTION_TSV, max_lines=100_000
)

print(f"Guarded passages : {stats_dev['kept']:,}")
print(f"Skipped lines    : {stats_dev['skipped']:,}")
print(f"Empty passages   : {stats_dev['empty']:,}")

Parsing: 100%|██████████| 100000/100000 [00:00<00:00, 1206275.40line/s]

Guarded passages : 100,000
Skipped lines    : 0
Empty passages   : 0


Complete parsing

In [18]:
if PID_MAP_FILE.exists():
    print(f"Chargement de la mapping existante : {PID_MAP_FILE}")
    docid_to_pid = np.load(PID_MAP_FILE)
    pid_to_docid = {int(pid): docid for docid, pid in enumerate(docid_to_pid)}
    stats = {'kept': len(docid_to_pid)}
    print(f"{stats['kept']:,} passages")
else:
    pid_to_docid, docid_to_pid, stats = parse_collection(COLLECTION_TSV)
    print(f"Guarded passages : {stats['kept']:,}")
    print(f"Skipped lines    : {stats['skipped']:,}")
    print(f"Empty passages   : {stats['empty']:,}")
    np.save(PID_MAP_FILE, docid_to_pid)
    print(f"Mapping saved : {PID_MAP_FILE}")

Chargement de la mapping existante : /Users/adrienkt/DataspellProjects/search-engine-trec-2020-deep-learning-track/work/data/pid_to_docid.npy
8,841,823 passages


Quick check

In [19]:
# Control : the 5 first passages (docid 0..4 ↔ pid via mapping)
def read_passage(docid):
    pid = int(docid_to_pid[docid])
    with open(COLLECTION_TSV, 'r', encoding='utf-8', errors='replace') as f:
        for i, line in enumerate(f):
            # simple scan ; in production, save an offset per docid
            if i == docid:
                parts = line.rstrip('\r\n').split('\t', 1)
                return pid, parts[1] if len(parts) == 2 else ''
    raise IndexError(docid)

for d in range(5):
    pid, text = read_passage(d)
    print(f"docid={d}  pid={pid}  text={text[:80]!r}")

docid=0  pid=0  text='The presence of communication amid scientific minds was equally important to the'
docid=1  pid=1  text='The Manhattan Project and its atomic bomb helped bring an end to World War II. I'
docid=2  pid=2  text='Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was'
docid=3  pid=3  text='The Manhattan Project was the name for a project conducted during World War II, '
docid=4  pid=4  text='versions of each volume as well as complementary websites. The first websiteâ\x80\x93T'


### 2.3 Loading queries and qrels with ir_datasets

In [20]:
import ir_datasets

DATASETS = {
    'trec-dl-2019':      'msmarco-passage/trec-dl-2019/judged',
    'trec-dl-2020':      'msmarco-passage/trec-dl-2020/judged',
    'msmarco-dev-small': 'msmarco-passage/dev/small',
}

def load_queries_and_qrels(name):
    ds = ir_datasets.load(DATASETS[name])
    queries = {q.query_id: q.text for q in ds.queries_iter()}
    qrels = [
        (q.query_id, q.doc_id, int(q.relevance))
        for q in ds.qrels_iter()
    ]
    return queries, qrels

for name in DATASETS:
    queries, qrels = load_queries_and_qrels(name)
    graded_levels = {r for _, _, r in qrels}          # set, ex. {0, 1, 2, 3}
    print(f"{name:20s}  queries={len(queries):>5}  qrels={len(qrels):>7}  "
          f"graded_levels={sorted(graded_levels) if len(graded_levels) < 10 else 'n/a'}")

trec-dl-2019          queries=   43  qrels=   9260  graded_levels=[0, 1, 2, 3]
trec-dl-2020          queries=   54  qrels=  11386  graded_levels=[0, 1, 2, 3]


[INFO] Please confirm you agree to the MSMARCO data usage agreement found at <http://www.msmarco.org/dataset.aspx>
[INFO] If you have a local copy of https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz, you can symlink it here to avoid downloading it again: /Users/adrienkt/.ir_datasets/downloads/31644046b18952c1386cd4564ba2ae69
[INFO] [starting] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz
[INFO] [finished] https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar


msmarco-dev-small     queries= 6980  qrels=   7437  graded_levels=[1]


In [21]:
def qrels_to_docids(qrels, pid_to_docid):
    """Convertit les qrels (query_id, pid, rel) en (query_id, docid, rel).
    Les pids absents de la collection (sous-ensemble dev) sont ignorés."""
    out = []
    missing = 0
    for qid, pid_str, rel in qrels:
        pid = int(pid_str)   # ir_datasets returns doc_id in str
        docid = pid_to_docid.get(pid)
        if docid is None:
            missing += 1
            continue
        out.append((qid, docid, rel))
    return out, missing

queries_20, qrels_20 = load_queries_and_qrels('trec-dl-2020')
qrels_20_docids, missing_20 = qrels_to_docids(qrels_20, pid_to_docid)
print(f"convert qrels : {len(qrels_20_docids)}  (missing pids : {missing_20})")

convert qrels : 11386  (missing pids : 0)


In [22]:
# Exemple : persister la mapping sur Drive depuis Colab
if IN_COLAB and DRIVE_MOUNT is not None:
    dst = DRIVE_MOUNT / 'AIDE' / 'MIR' / 'Search Engine TREC 2020 Deep Learning Track' / 'pid_to_docid.npy'
    if not dst.exists():
        import shutil
        shutil.copy(PID_MAP_FILE, dst)
        print(f"Mapping copy in Drive : {dst}")

## 3. Implementation

## 4. Tests

## 5. Stats

## 6. Evaluation and experiments

## 7. Report